# Policy-gradient agent - correctness checks

Run this notebook from the `part1/` folder so that `agent` is importable.

In [ ]:
import numpy as np
import torch
from agent import Policy, Agent, discount_rewards

S, A = 11, 3  # Hopper-v4 observation and action dims


## discount_rewards computes the discounted return

In [ ]:
out = discount_rewards(torch.tensor([1.0, 1.0, 1.0]), 0.9)
expected = torch.tensor([1 + 0.9 + 0.81, 1 + 0.9, 1.0])
assert torch.allclose(out, expected, atol=1e-5)
print("ok:", out.tolist())


## Policy output shapes and a strictly positive sigma

In [ ]:
dist, value = Policy(S, A)(torch.zeros(S))
assert dist.mean.shape == (A,)
assert value.shape == (1,)
assert torch.all(dist.stddev > 0)
print("ok: action mean", tuple(dist.mean.shape), "value", tuple(value.shape))


## get_action dimensions; evaluation returns the mean

In [ ]:
ag = Agent(Policy(S, A))
state = np.zeros(S, dtype=np.float32)
action, logp = ag.get_action(state)
assert action.shape == (A,) and logp is not None
mean_action, none = ag.get_action(state, evaluation=True)
assert none is None and mean_action.shape == (A,)
print("ok: sampled and deterministic actions have the right shape")


## One update step changes the parameters (REINFORCE and Actor-Critic)

In [ ]:
def fill(agent, n=8):
    for _ in range(n):
        s = np.zeros(S, dtype=np.float32)
        _, lp = agent.get_action(s)
        agent.store_outcome(s, s, lp, 1.0, False)

for actor_critic in (False, True):
    p = Policy(S, A)
    agent = Agent(p)
    before = [w.clone() for w in p.parameters()]
    fill(agent)
    agent.update_policy(baseline=0.0, actor_critic=actor_critic)
    assert any(not torch.equal(b, a) for b, a in zip(before, p.parameters()))
print("ok: parameters change for both REINFORCE and Actor-Critic")


## A baseline shrinks the policy-gradient weighting

In [ ]:
torch.manual_seed(0)
p = Policy(S, A)
agent = Agent(p)
fill(agent, 16)
rewards = torch.stack(agent.rewards).squeeze(-1)
logp = torch.stack(agent.action_log_probs).squeeze(-1)
G = discount_rewards(rewards, agent.gamma)
loss0 = -torch.sum(G * logp)
lossb = -torch.sum((G - G.mean()) * logp)
assert lossb.abs() <= loss0.abs() + 1e-4
print("ok: |loss| with baseline", round(lossb.abs().item(), 3), "<= without", round(loss0.abs().item(), 3))
